[Reference](https://medium.com/@inprogrammer/i-built-an-mcp-server-in-30-minutes-claude-and-cursor-could-use-my-api-immediately-f0592dcec24e$0)

```
uv init mcp-forecast-server
cd mcp-forecast-server
uv add "mcp[cli]" httpx
```

In [1]:
import os
from mcp.server.fastmcp import FastMCP
import httpx

mcp = FastMCP("forecast-server")
API_KEY = os.environ["FORECAST_API_KEY"]
BASE_URL = "https://api.myforecastservice.com/v1"
MAX_PAGES = 10  # guard against a stuck has_next_page flag
@mcp.tool()
async def get_forecast(city: str, days: int = 3) -> str:
    """Get the weather forecast for a city over a number of days.
    Args:
        city: Name of the city, e.g. "Tokyo" or "Bilaspur"
        days: How many days to forecast, between 1 and 7
    """
    headers = {"Authorization": f"Bearer {API_KEY}"}
    all_results = []
    page = 1
    async with httpx.AsyncClient() as client:
        while len(all_results) < days and page <= MAX_PAGES:
            response = await client.get(
                f"{BASE_URL}/forecast",
                headers=headers,
                params={"city": city, "page": page, "per_page": 5}
            )
            response.raise_for_status()
            data = response.json()
            all_results.extend(data["results"])
            if not data.get("has_next_page"):
                break
            page += 1
    lines = [
        f"{r['date']}: {r['summary']}, {r['temp']}°C"
        for r in all_results[:days]
    ]
    return f"Forecast for {city}:\n" + "\n".join(lines)
@mcp.tool()
async def get_alerts(city: str) -> str:
    """Get any active weather alerts for a city, e.g. storm or heat warnings."""
    headers = {"Authorization": f"Bearer {API_KEY}"}
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{BASE_URL}/alerts", headers=headers, params={"city": city})
        response.raise_for_status()
        data = response.json()
    if not data["alerts"]:
        return f"No active alerts for {city}."
    return "\n".join(a["description"] for a in data["alerts"])
if __name__ == "__main__":
    mcp.run(transport="stdio")